In [ ]:
# In[1]:  ── core imports & path setup ────────────────────────────────────────
import os, sys, torch, numpy as np, pandas as pd
from torchvision import transforms
from sklearn.model_selection import train_test_split

# Point to your project root (edit as required)
PATH = ""
# PATH = "storage"

# ---------------------------------------------------------------------------
#  Add any local repo paths you keep your custom loaders/datasets in
# ---------------------------------------------------------------------------
github_path = os.path.join(PATH, "Models", "GitHub", "EXAONEPath")
sys.path.append(github_path)

# HuggingFace login is only needed if you pull weights at runtime
# from huggingface_hub import login
# login(token="hf_your_token")

# ---------------------------------------------------------------------------
#  Device
# ---------------------------------------------------------------------------
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using:", device)

# ---------------------------------------------------------------------------
#  Import *updated* helper modules (they already contain evaluate_* func’s,
#  collect_model_stats(), run_all_model_stats(), and the new Evaluator class).
# ---------------------------------------------------------------------------
import reetoolbox.run_all_model_stats            as stats_mod   # our upgraded driver
import reetoolbox.eval_funcs          as rtf
# Ensure the updated Evaluator is on the import path
import reetoolbox.image_evaluator                as _check_     # just to confirm import



In [ ]:
from reetoolbox.dataloaders import *

In [ ]:
# dataset_name = "NCT"
# batch_size           = 8
# mode                 = 'tum_vs_all'
# test_multiplier      = 0.1
# root_dir            = os.path.join(PATH, 'data', 'NCT', 'CRC-VAL-HE-7K')
# kwargs = {"classification_mode": mode, }

# # NCT_dict = {
# #     "root_dir": test_root,
# #     "batch_size": batch_size,
# #     "test_multiplier": test_multiplier,
# # }

# dataclass = NCTDataSet
# test_data, test_loader = build_dataset(
#     NCTDataSet,
#     root_dir=root_dir,
#     batch_size=batch_size,
#     test_multiplier=test_multiplier,
#     transform=transforms.Compose([transforms.ToTensor()]),
#     classification_mode=mode,
#     shuffle=False
# )

# print(f"Test batches: {len(test_loader)}")

In [ ]:
dataset_name = "PanNuke"
root_dir = os.path.join(PATH, 'data', dataset_name)
batch_size = 8
test_multiplier = 0.285
kwargs = {"folds": (3,),
    "min_positive": 5,}

dataclass = PanNukeDataset
test_data, test_loader = build_dataset(
    PanNukeDataset,
    root_dir=root_dir,
    batch_size=batch_size,
    test_multiplier=test_multiplier,
    transform=transforms.Compose([transforms.ToTensor()]),
    folds=(3,),
    min_positive=5,
    shuffle=False
)

print(f"Test batches: {len(test_loader)}")

In [ ]:
# dataset_name = "PANDA"

# panda_root = os.path.join(PATH, 'data', dataset_name)
# root_dir = panda_root
# batch_size = 8
# test_multiplier = 0.097
# kwargs = {"split": "test"}

# dataclass = PandaDataset
# test_data, test_loader = build_dataset(
#     PandaDataset, 
#     root_dir = panda_root, 
#     batch_size = batch_size, 
#     test_multiplier = test_multiplier, 
#     transform=transforms.Compose([transforms.ToTensor()]), 
#     split='test'
# )

# print(f"Test batches: {len(test_loader)}")

In [ ]:
# dataset_name = "PatchCamelyon"

# patch_root    = os.path.join(PATH, 'data', 'PatchCamelyon')
# root_dir = patch_root
# batch_size    = 8
# test_multiplier = 0.022
# kwargs = {"split": "test"}

# dataclass = PatchCamelyonDataset
# test_data, test_loader = build_dataset(
#     PatchCamelyonDataset,
#     root_dir = patch_root,
#     batch_size = batch_size,
#     test_multiplier = test_multiplier,
#     transform=transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor()]),
#     split = 'test'
# )

# print(f"Test batches: {len(test_loader)}")

In [ ]:
# In[4]:  ── class balance check (unchanged) ─────────────────────────────────
counts = torch.zeros(2, dtype=torch.long)
for _, labels, _ in test_loader:
    counts += torch.bincount(labels, minlength=2)
print("Class counts:", counts.tolist())

In [ ]:
# In[5]:  ── model roster (identical to original, edit paths as needed) ──────
from reetoolbox.loadmodels import (
    load_uni, load_uni2, load_gigapath, load_virchow, load_virchow2,
    load_h0_mini, load_h_optimus0, load_h_optimus1,
    load_exaonepath, load_hibou_b, load_hibou_l,
    load_phikon, load_phikon_v2,
)

model_evals = [
    # ---- ResNet’s --------------------------------------------------------
    dict(
        model_name="ResNet18",
        load_func=lambda: torch.load(
            os.path.join(PATH, "Models", dataset_name, f"resnet18_{dataset_name}.pth"),
            weights_only=False,
        ),
        weight_path=None,
        output_subdir="ResNet18",
    ),
    dict(
        model_name="ResNet50",
        load_func=lambda: torch.load(
            os.path.join(PATH, "Models", dataset_name, f"resnet50_{dataset_name}.pth"),
            weights_only=False,
        ),
        weight_path=None,
        output_subdir="ResNet50",
    ),
    # ---- UniNet’s --------------------------------------------------------
    dict(
        model_name="UNI",
        load_func=lambda: load_uni(n_classes=2)[0],
        weight_path=os.path.join(PATH, "Models", dataset_name, f"uni_{dataset_name}.pth"),
        output_subdir="UNI",
    ),
    dict(
        model_name="UNI2",
        load_func=lambda: load_uni2(n_classes=2)[0],
        weight_path=os.path.join(PATH, "Models", dataset_name, f"uni2_{dataset_name}.pth"),
        output_subdir="UNI2",
    ),
    # ---- GigaPath --------------------------------------------------------
    dict(
        model_name="GigaPath",
        load_func=lambda: load_gigapath(n_classes=2)[0],
        weight_path=os.path.join(PATH, "Models", dataset_name, f"gigapath_{dataset_name}.pth"),
        output_subdir="GigaPath",
    ),
    # ---- Virchow family ---------------------------------------------------
    dict(
        model_name="Virchow",
        load_func=lambda: load_virchow(n_classes=2, device=device)[0],
        weight_path=os.path.join(PATH, "Models", dataset_name, f"virchow_{dataset_name}.pth"),
        output_subdir="Virchow",
    ),
    dict(
        model_name="Virchow2",
        load_func=lambda: load_virchow2(n_classes=2, device=device)[0],
        weight_path=os.path.join(PATH, "Models", dataset_name, f"virchow2_{dataset_name}.pth"),
        output_subdir="Virchow2",
    ),
    # ---- H0 and H-Optimus --------------------------------------------------
    dict(
        model_name="H0-mini",
        load_func=lambda: load_h0_mini(n_classes=2, device=device)[0],
        weight_path=os.path.join(PATH, "Models", dataset_name, f"h0_mini_{dataset_name}.pth"),
        output_subdir="H0-mini",
    ),
    dict(
        model_name="H-Optimus-0",
        load_func=lambda: load_h_optimus0(n_classes=2, device=device)[0],
        weight_path=os.path.join(PATH, "Models", dataset_name, f"hoptimus0_{dataset_name}.pth"),
        output_subdir="H-Optimus-0",
    ),
    dict(
        model_name="H-Optimus-1",
        load_func=lambda: load_h_optimus1(n_classes=2, device=device)[0],
        weight_path=os.path.join(PATH, "Models", dataset_name, f"hoptimus1_{dataset_name}.pth"),
        output_subdir="H-Optimus-1",
    ),
    # ---- EXAONEPath -------------------------------------------------------
    dict(
        model_name="EXAONEPath",
        load_func=lambda: load_exaonepath(n_classes=2, device=device)[0],
        weight_path=os.path.join(PATH, "Models", dataset_name, f"exaonepath_{dataset_name}.pth"),
        output_subdir="EXAONEPath",
    ),
    # ---- Hibou ------------------------------------------------------------
    dict(
        model_name="Hibou-B",
        load_func=lambda: load_hibou_b(n_classes=2, device=device)[1],
        weight_path=os.path.join(PATH, "Models", dataset_name, f"hibou_b_{dataset_name}.pth"),
        output_subdir="Hibou-B",
    ),
    dict(
        model_name="Hibou-L",
        load_func=lambda: load_hibou_l(n_classes=2, device=device)[1],
        weight_path=os.path.join(PATH, "Models", dataset_name, f"hibou_l_{dataset_name}.pth"),
        output_subdir="Hibou-L",
    ),
    # ---- Phikon -----------------------------------------------------------
    dict(
        model_name="Phikon v1",
        load_func=lambda: load_phikon(n_classes=2, device=device)[1],
        weight_path=os.path.join(PATH, "Models", dataset_name, f"phikon_{dataset_name}.pth"),
        output_subdir="Phikon_v1",
    ),
    dict(
        model_name="Phikon v2",
        load_func=lambda: load_phikon_v2(n_classes=2, device=device)[1],
        weight_path=os.path.join(PATH, "Models", dataset_name, f"phikon_v2_{dataset_name}.pth"),
        output_subdir="Phikon_v2",
    ),
]

In [ ]:
runs = 3

In [ ]:
stats_mod.run_all_model_stats(
    model_evals   = model_evals,
    root_dir      = root_dir,
    # root_dir      = os.path.join(PATH, 'data', dataset_name),
    batch_size    = batch_size,
    test_multiplier = test_multiplier,
    runs          = runs,
    device        = device,
    PATH          = PATH,
    save_dir_base = os.path.join(PATH, "Stats", dataset_name),
    dataset_class = dataclass,
    dataset_kwargs= kwargs,
)